## KNN&逻辑回归

In [ ]:
import pandas as pd           # 数据处理
import numpy as np            # 数学计算
from sklearn.model_selection import train_test_split  # 数据集划分
from sklearn.preprocessing import StandardScaler       # 数据标准化

# 加载数据
df = pd.read_csv('..//..//weather_data.csv')

# 查看数据基本信息
print("数据形状:", df.shape)
print("\n列名:")
print(df.columns.tolist())
print("\n数据类型:")
print(df.dtypes)
print("\n缺失值:")
print(df.isnull().sum())

In [ ]:
# 创建目标变量：是否有灾害
disaster_columns = ['frost_day', 'heat_day', 'severe_heat_day', 'dry_day',
                    'strong_wind_day', 'dust_storm_risk', 'rainy_day']
df['has_disaster'] = (df[disaster_columns].sum(axis=1) >= 3).astype(int)#同时两个发生定义为有害天气

# 选择特征
feature_columns = [
    'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
    'wind_speed_10m_max', 'relative_humidity_2m_mean'
]

# 创建衍生特征
df['temp_range'] = df['temperature_2m_max'] - df['temperature_2m_min']
df['temp_avg'] = (df['temperature_2m_max'] + df['temperature_2m_min']) / 2

# 更新特征列表
feature_columns.extend(['temp_range', 'temp_avg'])

# 准备数据
X = df[feature_columns].values   # 特征（输入）
y = df['has_disaster'].values     # 标签（输出）

print(f"\n特征维度: {X.shape[1]}")
print(f"样本数量: {X.shape[0]}")
print(f"灾害比例: {y.mean():.2%}")

In [ ]:
# 划分训练集和测试集
#X = df[feature_columns].values   # 366行，7列的数组
#y = df['has_disaster'].values     # 366个0或1

# 第2步：划分数据集
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20%作为测试集
    random_state=42,    # 随机种子
    stratify=y          # 保持训练集和测试集的类别比例相同
)

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n训练集: {X_train.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")

In [ ]:
"""
KNN


from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# 数据准备
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# KNN分类
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
# 评估
print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))
逻辑回归


from sklearn.linear_model import LogisticRegression
# 逻辑回归分类
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
# 评估
print(f"准确率: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

"""

class KNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = []
        for x in X:
            # 计算距离
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            # (self.X_train - x) 每个训练数据减去新数据
            # ** 2 平方
            # sum(axis=1) 求和
            # sqrt 开根号
            # 获取k个最近邻的索引
            k_indices = np.argsort(distances)[:self.k]
            # 获取对应的标签
            k_labels = self.y_train[k_indices]
            # 投票
            prediction = np.bincount(k_labels).argmax()
            predictions.append(prediction)
        return np.array(predictions)
class LogisticRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None

    def _sigmoid(self, z):
        # Sigmoid函数：把任意数字转换成0-1之间的概率
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))
        # np.clip 防止数值溢出

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        # 梯度下降
        for _ in range(self.n_iterations):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._sigmoid(linear_model)

            # 计算梯度
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # 更新参数
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X, threshold=0.5):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)
        return (y_predicted >= threshold).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

knn = KNN(k=9)
lr = LogisticRegression(learning_rate=0.01, n_iterations=100)
knn.fit(X_train_scaled, y_train)
lr.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_pred_lr = lr.predict(X_test_scaled)
print("训练完成！")
print('KNN 准确率:', accuracy_score(y_test, y_pred_knn))
print('逻辑回归 准确率:', accuracy_score(y_test, y_pred_lr))
print('KNN 精度:', precision_score(y_test, y_pred_knn))
print('逻辑回归 精度:', precision_score(y_test, y_pred_lr))
print('KNN 召回率:', recall_score(y_test, y_pred_knn))
print('逻辑回归 召回率:', recall_score(y_test, y_pred_lr))
print('KNN F1:', f1_score(y_test, y_pred_knn))
print('逻辑回归 F1:', f1_score(y_test, y_pred_lr))
print('KNN 混淆矩阵:\n', confusion_matrix(y_test, y_pred_knn))
print('逻辑回归 混淆矩阵:\n', confusion_matrix(y_test, y_pred_lr))  

'''
TN      FP

FN      TP
'''
